# 02 (Kaggle) — Resume/continue M0 baseline training

Kaggle replacement for `notebooks/02_train.ipynb`'s Colab flow: same
`src/train/sft.py`, same resumable-checkpoint mechanism, but persistent
storage is **Kaggle Datasets** instead of a mounted Google Drive folder
(Kaggle sessions don't keep `/kaggle/working` across sessions — only a
committed Dataset version does).

Before running this:
1. Attach two Kaggle Datasets as notebook inputs (Add Data, right panel):
   - `verilog-slm-data` — `corpus.jsonl`, `verilogeval_v2.jsonl`, `rtllm_v2.jsonl`
   - `verilog-slm-ckpt` — the latest `checkpoint-<N>/` folder (adapter + `trainer_state.pt`).
     Omit this input entirely if starting M0 fresh with no checkpoint to resume.
2. Notebook settings (right panel): Accelerator = **GPU T4 x2** (or P100),
   **Internet = On** (needed for `git clone`, `pip install`, `apt-get`, and
   the verible release download).
3. No dataset-slug variables to set -- the bootstrap cell searches all of
   `/kaggle/input` recursively by filename, since Kaggle's actual mount
   path for an attached dataset varies (`/kaggle/input/<slug>/` in some
   cases, `/kaggle/input/datasets/<user>/<slug>/` in others).

At the end of the session (see the last cell): **Save Version** so
`artifacts/m0/checkpoint-<N>/` becomes this notebook's Output, then push
that Output as a new version of `verilog-slm-ckpt` for the next session.

In [ ]:
# --- Kaggle-native bootstrap (replaces Colab's Drive-mount cell) ---
import os, shutil, glob

REPO = "https://github.com/saiswaroop25-pixel/verilog-slm"
os.chdir('/kaggle/working')
if not os.path.exists('/kaggle/working/verilog-slm'):
    os.system(f'git clone {REPO} /kaggle/working/verilog-slm')
os.chdir('/kaggle/working/verilog-slm')
os.system('git pull')

os.makedirs('artifacts', exist_ok=True)
os.makedirs('data/eval', exist_ok=True)
os.makedirs('artifacts/m0', exist_ok=True)

# Search all of /kaggle/input rather than building a path from a dataset
# slug: Kaggle's actual mount point for an attached dataset has been seen
# both as /kaggle/input/<slug>/ and /kaggle/input/datasets/<user>/<slug>/,
# and isn't worth depending on -- filename/dirname matching already
# disambiguates regardless of nesting depth or which convention applies.
def _find(filename):
    for p in glob.glob(f'/kaggle/input/**/{filename}', recursive=True):
        return p
    return None

for fname, dest in [
    ('corpus.jsonl', 'artifacts/corpus.jsonl'),
    ('verilogeval_v2.jsonl', 'data/eval/verilogeval_v2.jsonl'),
    ('rtllm_v2.jsonl', 'data/eval/rtllm_v2.jsonl'),
]:
    src = _find(fname)
    if src:
        shutil.copy(src, dest)
        print(f'restored {dest} <- {src}')
    else:
        print(f'WARNING: {fname} not found under /kaggle/input')

found = [p for p in glob.glob('/kaggle/input/**/checkpoint-*', recursive=True) if os.path.isdir(p)]
if not found:
    print('WARNING: no checkpoint-*/ found under /kaggle/input -- starting M0 from scratch')
for ckpt_dir in found:
    dest = f"artifacts/m0/{os.path.basename(ckpt_dir)}"
    if not os.path.exists(dest):
        shutil.copytree(ckpt_dir, dest)
        print(f'restored {dest} <- {ckpt_dir}')


In [ ]:
# Pinned deps, same as the Colab notebook.
!pip install -q -r requirements.txt -r requirements-train.txt

In [ ]:
import torch
print('torch:', torch.__version__, '| CUDA:', torch.version.cuda, '| GPU available:', torch.cuda.is_available())
!nvidia-smi -L

In [ ]:
# RTL toolchain: iverilog required, yosys/verible optional (soft-gated --
# see docs/industry_standards.md). Same install as the Colab notebook.
import os
!apt-get -qq update && apt-get -qq install -y iverilog yosys > /dev/null

!VERIBLE_URL=$(curl -s https://api.github.com/repos/chipsalliance/verible/releases/latest \
  | grep -o '"browser_download_url": *"[^"]*linux-static-x86_64.tar.gz"' \
  | head -1 | cut -d'"' -f4) && echo "resolved verible URL: $VERIBLE_URL" && \
  curl -sL "$VERIBLE_URL" -o /tmp/verible.tar.gz && \
  mkdir -p /opt/verible && tar -xzf /tmp/verible.tar.gz -C /opt/verible --strip-components=1

os.environ['PATH'] += ':/opt/verible/bin'
!iverilog -V | head -1
!yosys -V
!verible-verilog-lint --version

## Resume M0 baseline

`configs/m0_baseline_kaggle.yaml` is identical to `configs/m0_baseline.yaml`
(same `run_name: m0`, so it resumes the checkpoint just restored above) plus
`training.max_wall_hours: 11.5` -- `src/train/sft.py` checks this every step
and exits on a clean, resumable checkpoint before Kaggle's 12h hard session
cap kills the process outright.

In [ ]:
!python -m src.train.sft --config configs/m0_baseline_kaggle.yaml 2>&1 | tee artifacts/m0_stdout.log

## End of session -- carry the checkpoint forward

1. Confirm the run stopped on a clean checkpoint: last line of the cell
   above should be either the wall-clock-budget message or full completion,
   not a truncated stack trace.
2. **Save Version** (top right) to commit `/kaggle/working` as this
   notebook's Output.
3. Push that Output as a new **version** of the `verilog-slm-ckpt` dataset
   (or attach the notebook's own Output directly as input next time).
4. Next session: copy this notebook, re-run from the top -- the bootstrap
   cell restores the new checkpoint and training resumes from there.